# EKD Paper Evidential Teacher ResNet-110


## Imports and Configuration


In [1]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


KAGGLE_WORKING = Path("/kaggle/working")
BASE_OUTPUT_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "outputs"

CONFIG = {
    "paper": "Evidential Knowledge Distillation, ICCV 2025",
    "official_config": "configs/cifar100/teacher/ResNet110.yaml",
    "method": "Evidential Teacher",
    "model": "resnet110",
    "dataset": "CIFAR100",
    "num_classes": 100,
    "epochs": 240,
    "batch_size": 64,
    "test_batch_size": 64,
    "optimizer": "SGD",
    "learning_rate": 0.05,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "scheduler": "MultiStepLR",
    "milestones": [150, 180, 210],
    "gamma": 0.1,
    "teacher_lamb_config": 0.0,
    "teacher_lamb_init": 1.0,
    "teacher_lamb_is_learnable": True,
    "evidence_function": "exp",
    "loss": "Evidential Cross Entropy",
    "seed": 42,
    "num_workers": 2,
    "output_root": str(BASE_OUTPUT_DIR / "kd_cifar100_artifacts"),
    "stage_dir": "ekd_paper_teacher_resnet110",
}


In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


## Model Building Block Helper


In [3]:
def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(
        in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False
    )


## Class: `BasicBlock`


In [4]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, is_last=False):
        super().__init__()
        self.is_last = is_last
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out = out + residual
        preact = out
        out = F.relu(out, inplace=True)
        if self.is_last:
            return out, preact
        return out


## Class: `CIFARResNet`


In [5]:
class CIFARResNet(nn.Module):
    def __init__(self, depth, num_filters, num_classes=100):
        super().__init__()
        assert (depth - 2) % 6 == 0
        n = (depth - 2) // 6
        self.inplanes = num_filters[0]
        self.conv1 = nn.Conv2d(3, num_filters[0], kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_filters[0])
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(BasicBlock, num_filters[1], n)
        self.layer2 = self._make_layer(BasicBlock, num_filters[2], n, stride=2)
        self.layer3 = self._make_layer(BasicBlock, num_filters[3], n, stride=2)
        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(num_filters[3], num_classes)
        self.stage_channels = num_filters
        self._initialize_weights()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.inplanes,
                    planes * block.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(planes * block.expansion),
            )
        layers = [block(self.inplanes, planes, stride, downsample, is_last=(blocks == 1))]
        self.inplanes = planes * block.expansion
        for idx in range(1, blocks):
            layers.append(block(self.inplanes, planes, is_last=(idx == blocks - 1)))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        f0 = x
        x, f1_pre = self.layer1(x)
        f1 = x
        x, f2_pre = self.layer2(x)
        f2 = x
        x, f3_pre = self.layer3(x)
        f3 = x
        x = self.avgpool(x)
        pooled = x.reshape(x.size(0), -1)
        logits = self.fc(pooled)
        feats = {
            "feats": [f0, f1, f2, f3],
            "preact_feats": [f0, f1_pre, f2_pre, f3_pre],
            "pooled_feat": pooled,
        }
        return logits, feats


## Model Factories


In [6]:
def resnet110(num_classes=100):
    return CIFARResNet(110, [16, 16, 32, 64], num_classes=num_classes)


## Class: `EvidentialTeacher`


In [7]:
class EvidentialTeacher(nn.Module):
    def __init__(self, backbone, lamb_config=0.0, lamb_init=1.0):
        super().__init__()
        self.student = backbone
        if lamb_config == 0.0:
            self.lamb = nn.Parameter(torch.tensor(lamb_init, dtype=torch.float32))
        else:
            self.register_buffer("lamb", torch.tensor(lamb_config, dtype=torch.float32))

    def get_learnable_parameters(self):
        params = []
        if isinstance(self.lamb, nn.Parameter):
            params.append(self.lamb)
        params.extend([param for param in self.student.parameters()])
        return params

    def forward_train(self, images, labels):
        logits, _ = self.student(images)
        evidence = torch.exp(logits)
        alpha = evidence + torch.exp(self.lamb)
        labels_1hot = torch.zeros_like(logits).scatter_(-1, labels.unsqueeze(-1), 1)
        strength = torch.sum(alpha, dim=-1, keepdim=True)
        loss_ce = torch.sum(
            labels_1hot * (torch.digamma(strength) - torch.digamma(alpha)), dim=-1
        ).mean()
        return logits, loss_ce

    def forward(self, images):
        logits, _ = self.student(images)
        evidence = torch.exp(logits)
        return evidence


## CIFAR-100 Data Loaders


In [8]:
def build_loaders():
    mean = (0.5071, 0.4867, 0.4408)
    std = (0.2675, 0.2565, 0.2761)
    train_transform = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    test_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean, std)]
    )
    data_root = KAGGLE_WORKING / "data" if KAGGLE_WORKING.exists() else Path("./data")
    train_set = datasets.CIFAR100(
        root=str(data_root), train=True, download=True, transform=train_transform
    )
    test_set = datasets.CIFAR100(
        root=str(data_root), train=False, download=True, transform=test_transform
    )
    train_loader = DataLoader(
        train_set,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_set,
        batch_size=CONFIG["test_batch_size"],
        shuffle=False,
        num_workers=1,
        pin_memory=torch.cuda.is_available(),
    )
    return train_set, test_set, train_loader, test_loader


## Metrics


In [9]:
def topk_accuracy(outputs, labels, topk=(1, 5)):
    maxk = max(topk)
    _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)
    pred = pred.t()
    correct = pred.eq(labels.reshape(1, -1).expand_as(pred))
    results = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0)
        results.append((correct_k * (100.0 / labels.size(0))).item())
    return results


## Evaluation


In [10]:
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total = 0
    criterion = nn.CrossEntropyLoss()
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        top1, top5 = topk_accuracy(outputs, labels, topk=(1, 5))
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_top1 += top1 * batch_size
        total_top5 += top5 * batch_size
        total += batch_size
    return total_loss / total, total_top1 / total, total_top5 / total


## Checkpointing


In [11]:
def save_checkpoint(path, epoch, wrapper, optimizer, scheduler, best_acc, config, train_set):
    checkpoint = {
        "epoch": epoch,
        "model_name": config["model"],
        "method": config["method"],
        "model_state_dict": wrapper.student.state_dict(),
        "wrapper_state_dict": wrapper.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_acc": best_acc,
        "lamb": float(wrapper.lamb.detach().cpu()),
        "config": config,
        "class_to_idx": train_set.class_to_idx,
        "classes": train_set.classes,
    }
    torch.save(checkpoint, path)


## Plotting


In [12]:
def plot_curves(log_df, plot_dir):
    plt.figure(figsize=(8, 5))
    plt.plot(log_df["epoch"], log_df["train_acc"], label="Train Accuracy")
    plt.plot(log_df["epoch"], log_df["test_acc"], label="Test Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("EKD Paper Evidential Teacher Accuracy")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / "evidential_teacher_accuracy_curve.png", dpi=200)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(log_df["epoch"], log_df["train_loss"], label="Train Evidential CE")
    plt.plot(log_df["epoch"], log_df["test_loss"], label="Test CE on Evidence")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("EKD Paper Evidential Teacher Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / "evidential_teacher_loss_curve.png", dpi=200)
    plt.close()


## Training Pipeline


In [13]:
def main():
    set_seed(CONFIG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    stage_root = Path(CONFIG["output_root"]) / CONFIG["stage_dir"]
    checkpoint_dir = stage_root / "checkpoints"
    log_dir = stage_root / "logs"
    config_dir = stage_root / "config"
    plot_dir = stage_root / "plots"
    for directory in [checkpoint_dir, log_dir, config_dir, plot_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    with open(config_dir / "ekd_paper_teacher_resnet110_config.json", "w") as f:
        json.dump(CONFIG, f, indent=4)

    train_set, _, train_loader, test_loader = build_loaders()
    backbone = resnet110(num_classes=CONFIG["num_classes"])
    model = EvidentialTeacher(
        backbone,
        lamb_config=CONFIG["teacher_lamb_config"],
        lamb_init=CONFIG["teacher_lamb_init"],
    ).to(device)
    optimizer = optim.SGD(
        model.get_learnable_parameters(),
        lr=CONFIG["learning_rate"],
        momentum=CONFIG["momentum"],
        weight_decay=CONFIG["weight_decay"],
    )
    scheduler = MultiStepLR(
        optimizer, milestones=CONFIG["milestones"], gamma=CONFIG["gamma"]
    )

    best_acc = -1.0
    records = []
    start_time = time.time()
    best_path = checkpoint_dir / "ekd_paper_teacher_resnet110_best.pth"
    last_path = checkpoint_dir / "ekd_paper_teacher_resnet110_last.pth"
    log_path = log_dir / "ekd_paper_teacher_training_log.csv"

    for epoch in range(1, CONFIG["epochs"] + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0
        total_top1 = 0.0
        total_top5 = 0.0
        total = 0
        current_lr = optimizer.param_groups[0]["lr"]

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True).float()
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits, loss = model.forward_train(images, labels)
            loss.backward()
            optimizer.step()

            top1, top5 = topk_accuracy(logits, labels, topk=(1, 5))
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_top1 += top1 * batch_size
            total_top5 += top5 * batch_size
            total += batch_size

        train_loss = total_loss / total
        train_acc = total_top1 / total
        train_acc_top5 = total_top5 / total
        test_loss, test_acc, test_acc_top5 = evaluate(model, test_loader, device)
        scheduler.step()

        if test_acc >= best_acc:
            best_acc = test_acc
            save_checkpoint(best_path, epoch, model, optimizer, scheduler, best_acc, CONFIG, train_set)

        save_checkpoint(last_path, epoch, model, optimizer, scheduler, best_acc, CONFIG, train_set)

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_acc_top5": train_acc_top5,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "test_acc_top5": test_acc_top5,
            "best_test_acc": best_acc,
            "lamb": float(model.lamb.detach().cpu()),
            "exp_lamb": float(torch.exp(model.lamb.detach()).cpu()),
            "epoch_time_sec": time.time() - epoch_start,
        }
        records.append(record)
        pd.DataFrame(records).to_csv(log_path, index=False)
        print(
            f"Epoch {epoch:03d}/{CONFIG['epochs']} | lr {current_lr:.5f} | "
            f"train {train_loss:.4f}, acc {train_acc:.2f}% | "
            f"test {test_loss:.4f}, acc {test_acc:.2f}% | best {best_acc:.2f}% | "
            f"lamb {record['lamb']:.4f}"
        )

    log_df = pd.DataFrame(records)
    plot_curves(log_df, plot_dir)

    summary = {
        "method": CONFIG["method"],
        "model": "ResNet110",
        "dataset": "CIFAR100",
        "best_test_acc": best_acc,
        "final_test_acc": float(log_df.iloc[-1]["test_acc"]),
        "final_train_acc": float(log_df.iloc[-1]["train_acc"]),
        "params": sum(p.numel() for p in model.student.parameters()),
        "epochs": CONFIG["epochs"],
        "batch_size": CONFIG["batch_size"],
        "optimizer": CONFIG["optimizer"],
        "scheduler": CONFIG["scheduler"],
        "seed": CONFIG["seed"],
        "lamb": float(model.lamb.detach().cpu()),
        "exp_lamb": float(torch.exp(model.lamb.detach()).cpu()),
        "checkpoint_path": str(best_path),
        "last_checkpoint_path": str(last_path),
        "training_time_seconds": time.time() - start_time,
    }
    summary_path = stage_root / "ekd_paper_teacher_resnet110_summary.json"
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=4)
    print(json.dumps(summary, indent=4))


## Run Training

Run this cell to start the full paper-matched training job.


In [14]:
main()


Using device: cuda
GPU: Tesla T4


100%|██████████| 169M/169M [00:03<00:00, 50.4MB/s] 


Epoch 001/240 | lr 0.05000 | train 4.6075, acc 2.06% | test 862.9933, acc 2.61% | best 2.61% | lamb 4.7736
Epoch 002/240 | lr 0.05000 | train 4.5475, acc 3.62% | test 477.1054, acc 5.23% | best 5.23% | lamb 4.6535
Epoch 003/240 | lr 0.05000 | train 4.4222, acc 5.91% | test 1075.5606, acc 8.88% | best 8.88% | lamb 3.7471
Epoch 004/240 | lr 0.05000 | train 3.9805, acc 10.98% | test 10236.7044, acc 15.65% | best 15.65% | lamb 1.8526
Epoch 005/240 | lr 0.05000 | train 3.3320, acc 19.02% | test 117890.2662, acc 22.69% | best 22.69% | lamb 0.2496
Epoch 006/240 | lr 0.05000 | train 2.8682, acc 27.04% | test 164576.3894, acc 30.31% | best 30.31% | lamb -0.2169
Epoch 007/240 | lr 0.05000 | train 2.5519, acc 33.52% | test 4931342.0615, acc 29.93% | best 30.31% | lamb -0.4669
Epoch 008/240 | lr 0.05000 | train 2.3095, acc 38.49% | test 57978191.5146, acc 35.97% | best 35.97% | lamb -0.6410
Epoch 009/240 | lr 0.05000 | train 2.1381, acc 42.16% | test 31241889.2108, acc 39.88% | best 39.88% | lamb 

## Zip Results

Run this after training to package the notebook outputs.


In [15]:
import shutil
from pathlib import Path

stage_root = Path(CONFIG["output_root"]) / "ekd_paper_teacher_resnet110"
zip_base = BASE_OUTPUT_DIR / "ekd_paper_teacher_resnet110_artifacts"

if not stage_root.exists():
    print("Artifact directory does not exist yet:", stage_root)
    print("Run the training cell above before zipping outputs.")
else:
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=stage_root.parent, base_dir=stage_root.name)
    print("Created zip file:")
    print(zip_path)


Created zip file:
/kaggle/working/ekd_paper_teacher_resnet110_artifacts.zip
